# 🦾🧠 **MLOps on RHOAI: RedHat OpenShift AI 介绍与安装**

## 📑 **0. 目录**

- 🍾 1. RHOAI 的高层次 (High-level) 架构
- 🔥 2. RHOAI 中的 AI/ML 工作流示意
- 🧪 3. 部署 Red Hat OpenShift AI (RHOAI)
- 🔎 4. 参考链接

## 🍾 **1. RHOAI 的高层次 (High-level) 架构**

### **1.1 RedHat AI 生态图**

<center><img src="images/RedHat-AI-1.png" style="width:80%"></center>

<center><img src="images/RedHat-AI-2.jpg" style="width:80%"></center>

### **1.2 高级 RHOAI 组件架构**

<center><img src="images/01-rhoai-advanced-architecture.jpg" style="width:80%"></center>

<center>图例：高级 RHOAI 组件架构</center>

### **1.3 RHOAI 中的 workbench 工作台**

- RHOAI 中模型开发与训练位于 **workbench** 中进行，它在整个 MLOps 中具有核心的地位，workbench 以 `Pod` 的形式处于核心地位。
- 以下列举了 workbench 的镜像来源与架构：
  
<center><img src="images/workbench-architecture.png" style="width:60%"></center>

<center>图例：workbench 架构</center>

### **1.4 RHOAI 基于的上游项目**

- 数据科学管道项目（Data Science Pipelines）基于 `Tekton` 与 [Kubeflow](https://www.kubeflow.org/) 上游项目，可使用 `Elyra` GUI 可视化创建管道。
- RHOAI 基于开放数据中心（[Open Data Hub](https://opendatahub.io/), `ODH`）上游项目

<center><img src="images/open-data-hub-arch.png" style="width:80%"></center>

<center>图例：Open Data Hub（ODH）项目</center>

  - **Open Data Hub Dashboard**：Web 仪表板，显示已安装的开放数据中心组件，易于访问组件 UI 和文档。
  - **ODH Notebook Controller**：Kubernetes 环境中 Jupyter Notebook 的安全管理，建立在 Kubeflow Notebook Controller 之上，支持 OAuth。
  - **Jupyter Notebooks**：为 GPU 工作负载提供 Python 支持的 JupyterLab notebook。
  - **Data Science Pipelines**：支持 Kubeflow Pipeline SDK 和 Tekton 的端到端 MLOps 工作流的管道解决方案。
  - **Model Mesh**：ModelMesh 服务是用于管理 ModelMesh 的控制器，ModelMesh 是一个通用的模型，服务于管理/路由层。

## 🔥 **2. RHOAI 中的 AI/ML 工作流示意**

<center><img src="images/02-ml-workflow-in-rhoai.png" style="width:80%"></center>

<center>图例：RedHat OpenShift 中的机器学习工作流</center>

- RHOAI 中的组件通过 **Operator** 的方式进行部署安装
- RHOAI GPU 功能需要 **NVIDIA GPU Operator**
- RHOAI 工作台（workbench）作为 OpenShift Pod 运行，专为机器学习和数据科学而设计。
- 为此，RHOAI 提供工作台镜像，以便使用 `TensorFlow`、`PyTorch` 和 `Scikit-learn` 等常用库来训练模型。
- 工作台包含 JupyterLab Notebook 执行环境、标准数据科学库（如 Standard Data Science、TensorFlow、PyTorch）和 GPU 加速功能等。
  > 注意：JupyterLab 的前身被称为 Jupyter Notebook。
- 在 RHOAI 中，数据科学管道是以自动化方式执行脚本或 Jupyter Notebook 的工作流。
- AI 模型、机器学习模型（或简称模型）是机器学习工作流训练阶段产生的主要制品（Artifacts）
- 通常，开发人员还需要 **REST** 或 **gRPC API**，以便面向公众公开模型，**具体采用哪种方式公开要取决于模型服务器（model server）的支持**。
- 在 RHOAI 中，模型服务器是自动部署模型的组件。
- 💪 RHOAI 使用 [KServe](https://kserve.github.io/website) 作为 **模型服务平台（model serving platform）**，并支持 `OpenVINO`、`Triton`、文本生成推断服务器（`TGIS`）和 `Caikit` 等模型运行时。
- 模型服务器使用数据连接（data connection）从 S3 模型存储中下载模型文件
  > 注意：上图中已标注使用 S3 进行数据集加载、模型文件上传与模型文件下载的执行点
- 将模型文件下载到运行模型服务器的容器中后，模型服务器会通过标准 REST 或 gRPC API 公开模型。

## 🧪 **3. 部署 Red Hat OpenShift AI (RHOAI)**

- RHOAI Operator 是 RHOAI 的主要组件，部署与管理所有依赖的 Operator 与 组件。
- RHOAI 部署涉及的 Operator 如下所示：
  - 1️⃣ **RHOAI operator**
  - 2️⃣ **Red Hat OpenShift Pipelines operator**
  - 3️⃣ **Red Hat OpenShift Serverless & Red Hat OpenShift Service Mesh operators**
  - 4️⃣ **NVIDIA GPU and Node Feature Discovery operators**
- RHOAI 通过 Operator 与自定义资源定义（CRD）的方式实现自定义资源（CR）
- 相关的自定义资源定义包括：
  - 1️⃣ **DataScienceCluster CRD (DSC)**：
    - CRD 全称：`datascienceclusters.datasciencecluster.opendatahub.io`
    - 功能：负责创建 FeatureTracker 对象与 DSCInitialization 对象
  - 2️⃣ **FeatureTracker CRD**：
    - CRD 全称：`featuretrackers.feature.opendatahub.io`
    - 功能：负责保持对 Operator 创建的 OpenShift 资源的引用，以实现未使用的资源对象的垃圾回收。
  - 3️⃣ **DSCInitialization CRD**：
    - CRD 全称：`dscinitializations.dscinitialization.opendatahub.io`
    - 功能：负责验证需要存在的所有 Kubernetes 对象，如 Namespace、ConfigMap、NetworkPolicy、Service、Role 等。
- 因此，OpenShift 管理员只需创建 DataScienceCluster 资源对象即可，且一个集群中只能定义一个此资源对象。
- 可在 DataScienceCluster 对象资源定义中选择组件的状态，即 Removed 或 Managed。

### 🌈 **3.1 RHOAI Operator 安装与就绪时序**

- 1️⃣ RHOAI Operator 安装的过程（Dashboard 或命令行）实际上是在 redhat-ods-operator 命名空间中部署 `rhods-operator deployment` 及其他资源对象。若采用 RHOAI Dashboard 部署的 Operator，在显示安装完成后，仅在此命名空间中创建资源对象。
- 2️⃣ Operator 部署完成后还需修改名为 default-dsc 的 DataScienceCluster 对象，根据所需部署组件更新 Removed 或 Managed，一旦更新后，此对象将创建 DSCInitialization 对象与 FeatureTracker 对象。
- 3️⃣ 随后 redhat-ods-applications 与 redhat-ods-monitoring 命名空间自动创建。
- 4️⃣ 在 redhat-ods-applications 命名空间中逐渐创建出各个与 RHOAI 相关的资源对象，但在 Dashboard 的 default-dsc 对象的 Conditions 中显示 Serverless Operator 缺失而导致其状态异常。
- 5️⃣ 因此，安装完 Serverless Operator 后，default-dsc 对象状态从 Progressing 变为 Ready。redhat-ods-applications 命名空间中将生成所有相关的资源对象。

<center><img src="images/RHOAI-Operator-install-ready-sequence-chart.png" style="width:80%"></center>

<center>图例：RHOAI Operator 安装与就绪时序图</center>

<center><img src="images/redhat-ods-applications-pods.png" style="width:80%"></center>

<center>图例：RHOAI 部署完成后 redhat-ods-applications 命名空间中运行的 pod</center>

### **3.2 CRD 与 CR 对象查询**

💡 **CRD 定义新的资源对象类型，CR 是这个对象类型的实例，Operator 是监听并处理这些 CR 的控制器。**

<center><img src="images/RHOAI-operator-spec.png" style="width:80%"></center>

<center>图例：redhat-ods-operator 项目中的 3 类 CRD</center>

## 🔎 **4. 参考链接**

- [红帽 AI/ML 与 MLOps 客户成功故事](https://www.redhat.com/en/blog/red-hat-ai/ml-and-mlops-customer-success-stories)
- [**Knowledgebase: "Red Hat OpenShift AI: Supported Configurations"**](https://access.redhat.com/articles/rhoai-supported-configs)
- [**Knowledgebase: "How to deploy a machine learning model by using KServe
RawDeployment mode with single node OpenShift"**](https://access.redhat.com/solutions/7078183)
- [**Knowledgebase: "Red Hat OpenShift AI Service Definition"**](https://access.redhat.com/support/policy/updates/rhoai/service)